# Lesson 13 - Random Colour Ball Push

You have already used basic movement commands. You have also used the distance sensor to make the robot react to what is in front of it.

Now the camera is the sensor. The robot will look for a coloured ball and use the camera result to choose a movement.

Goal:
- calibrate the ball colours
- randomly choose one ball
- use the camera to decide left, right, center, or lost
- write the movement code
- tweak the movement times until the robot can push any ball

The robot should always sense first, then decide, then move.

## 1. Start The Robot

Run this first.

In [ ]:
from lesson_header import *
import random
import time

show_v2_status()
myRobot = bot(base_speed=300)
myRobot.camera.center_all()

## 2. Check The Camera

Put the balls in front of the robot. The picture should match the real left and right.

In [ ]:
picture = myRobot.vision.capture()
frame = picture["frame_bgr"]
print("frame size:", frame.shape)

## 3. Calibrate The Colours

Use Robot Ops Web to click each ball. Paste the lower and upper HSV values below.

These numbers are examples. Change them if your room lighting is different.

In [ ]:
myRobot.vision.set_color_profile(
    "blue",
    lower_hsv=(105, 83, 127),
    upper_hsv=(129, 223, 255),
)

myRobot.vision.set_color_profile(
    "green",
    lower_hsv=(47, 98, 98),
    upper_hsv=(71, 238, 238),
)

myRobot.vision.set_color_profile(
    "red",
    lower_hsv=(165, 122, 184),
    upper_hsv=(179, 255, 255),
)

myRobot.vision.show_profiles()

## 4. Test Each Colour

Run these one at a time. The camera should highlight the correct ball.

In [ ]:
myRobot.vision.show_color("blue")
myRobot.vision.show_color("green")
myRobot.vision.show_color("red")

## 5. Pick A Random Ball

This wrapper chooses the target ball for the challenge.

In [ ]:
BALL_COLOURS = ["blue", "green", "red"]

def pick_random_ball():
    colour = random.choice(BALL_COLOURS)
    print("Target ball:", colour)
    return colour

TARGET_COLOUR = pick_random_ball()

## 6. Ask The Camera For A Decision

`target_position` does not move the robot. It only tells you what the camera sees.

The direction will be `left`, `right`, `center`, or `lost`.

In [ ]:
decision = myRobot.vision.target_position(
    TARGET_COLOUR,
    deadzone=50,
    show=True,
)

print("found:", decision["found"])
print("direction:", decision["direction"])
print("error from centre:", decision["error"])

## 7. Write One Movement Decision

This is the main algorithm. Run it, watch what happens, then change the seconds and speed values.

In [ ]:
decision = myRobot.vision.target_position(
    TARGET_COLOUR,
    deadzone=50,
    show=True,
)

print("direction:", decision["direction"])

if decision["direction"] == "left":
    myRobot.move.left(seconds=0.15, speed=80)

elif decision["direction"] == "right":
    myRobot.move.right(seconds=0.15, speed=80)

elif decision["direction"] == "center":
    myRobot.move.forward(seconds=0.6, speed=100)

elif decision["direction"] == "lost":
    print("I cannot see the", TARGET_COLOUR, "ball")

myRobot.move.stop()

## 8. Improve It With Small Moves

Now repeat the decision. The robot moves a little bit, checks again, and keeps going until it is centred.

Tune `SIDEWAYS_SECONDS` first.

In [ ]:
DEADZONE = 50
SIDEWAYS_SECONDS = 0.12
SIDEWAYS_SPEED = 80
PUSH_SECONDS = 0.6
PUSH_SPEED = 100
MAX_TRIES = 8

for attempt in range(MAX_TRIES):
    decision = myRobot.vision.target_position(
        TARGET_COLOUR,
        deadzone=DEADZONE,
        show=True,
    )

    print("try", attempt + 1, "direction:", decision["direction"], "error:", decision["error"])

    if decision["direction"] == "left":
        myRobot.move.left(seconds=SIDEWAYS_SECONDS, speed=SIDEWAYS_SPEED)

    elif decision["direction"] == "right":
        myRobot.move.right(seconds=SIDEWAYS_SECONDS, speed=SIDEWAYS_SPEED)

    elif decision["direction"] == "center":
        myRobot.move.forward(seconds=PUSH_SECONDS, speed=PUSH_SPEED)
        myRobot.move.stop()
        print("Pushed the", TARGET_COLOUR, "ball")
        break

    elif decision["direction"] == "lost":
        print("I cannot see the", TARGET_COLOUR, "ball")
        break

    myRobot.move.stop()
    time.sleep(0.2)

else:
    print("I tried", MAX_TRIES, "times, but did not centre the ball")

## Challenge

Make the robot push a random ball three times in a row.

Things to test:
- What happens if `SIDEWAYS_SECONDS` is too big?
- What happens if it is too small?
- What deadzone works best?
- What should the robot do when the ball is lost?